# K_04 – Saisonale Animationen

**Grid-Arbitrage** · Batteriespeicher-Arbitrage im Schweizer Strommarkt (Kür)

**Gruppe:** SC26_Gruppe_2 | **Verantwortlich:** Patrik Neunteufel | **Datum:** März 2026

---

*Animierter Jahresverlauf: Spread, Netzlast und Arbitrage-Fenster als GIF.*


| [← K_03 – Marktdynamik](K_03_Marktdynamik.ipynb) | [↑ Übersicht ↑](../organisation/O_01_Project_Overview.ipynb) | [K_05 – Revenue Stacking →](K_05_Revenue_Stacking.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_K_04'></a>

1. [Einleitung](#einleitung_K_04)
2. [Initialisierung](#initialisierung_K_04)
3. [Aggregation und Animations-Setup](#aggregation-und-animations-setup_K_04)
4. [Verifikation](#verifikation_K_04)
5. [Option A: Animation je Tageszeit (52 Wochen)](#option-a-animation-je-tageszeit-52-wochen_K_04)
6. [Option B: 4-Panel-Animation](#option-b-4-panel-animation_K_04)
7. [Option C: Spread-Animation rollierend](#option-c-spread-animation-rollierend_K_04)
8. [Fazit](#fazit_K_04)
9. [Potenzielle Erweiterungen](#erweiterungen_K_04)
10. [Abschluss](#abschluss_K_04)


---
## Einleitung <a id='einleitung_K_04'></a>

[↑ Inhaltsverzeichnis](#toc_K_04)

Saisonale Animationen aus den NB02-Preisdaten, drei Varianten:

- **Option A** — Animation je Tageszeit über 52 Wochen (zeigt wann Spread-
  Fenster im Jahresverlauf entstehen)
- **Option B** — 4-Panel-Animation mit verschiedenen Kennzahlen gleichzeitig
- **Option C** — Spread-Animation rollierend (12-Monate-Fenster)

Alle GIFs werden in `output/charts/` abgelegt. Parameter (FPS, DPI,
Frames/Woche, Einzelframe-Export) stammen aus `../sync/config.json`
`['animation']`. Cubic-Spline-Interpolation für flüssige Übergänge.


## Initialisierung<a id='initialisierung_K_04'></a>

[↑ Inhaltsverzeichnis](#toc_K_04)

Bibliotheken laden, `../sync/config.json` lesen, Verzeichnispfade setzen.

**Imports und Versionen:**

In [14]:
# ── lib/ aus Projekt-Root erreichbar machen + lib-Imports ───────────────────
# Notebook liegt in einem Unterordner (kuer/, experimental/, notebooks/,
# organisation/). Damit 'from lib.xxx import ...' funktioniert, muss der
# Projekt-Root vorne in sys.path stehen. autoreload sorgt dafür, dass
# Änderungen in lib/*.py ohne Kernel-Restart übernommen werden.
import sys, os
_PROJECT_ROOT = os.path.abspath('..')
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

# lib-Imports (einmal zentral — in allen folgenden Zellen verfügbar)
from lib.widgets  import show_animation
from lib.plotting import show_source, should_skip, make_gif_chart
from lib.io_ops   import load_transfer, final_check

print(f'lib-Pfad aktiv: {_PROJECT_ROOT}/lib')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
lib-Pfad aktiv: C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage/lib


In [15]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore')
from datetime import datetime
from matplotlib.animation import FuncAnimation, PillowWriter
import matplotlib.patches as mpatches
import math

# Versionen anzeigen für Reproduzierbarkeit
print(f"Numpy        Version: {np.__version__}")
print(f"Pandas       Version: {pd.__version__}")
print(f"📅 Zuletzt ausgeführt am: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")


Numpy        Version: 2.2.6
Pandas       Version: 2.3.3
📅 Zuletzt ausgeführt am: 26.04.2026 um 13:13:40


**Setup – Konfiguration & Verzeichnisstruktur:** Lädt `../sync/config.json` (SSOT), setzt Pfade.

In [16]:
with open('../sync/config.json') as _f:
    CFG = json.load(_f)

SZ_AKTIV     = CFG['szenarien']['gleichzeitigkeit_aktiv']
FORCE_RELOAD = CFG.get('force_reload', {})  # konventionskonform gelesen
DIR_RAW      = os.path.join('../data', 'raw')
DIR_PROC     = os.path.join('../data', 'processed')
DIR_INTER    = os.path.join('../data', 'intermediate')
DIR_INTER_SZ = os.path.join(DIR_INTER, SZ_AKTIV)
CHARTS_DIR   = os.path.join('../output', 'charts', SZ_AKTIV)
os.makedirs(CHARTS_DIR, exist_ok=True)
DPI      = CFG['visualisierung']['output_dpi']       # SSOT: ../sync/config.json

# ── Farben & Stil aus ../sync/config.json (SSOT) ─────────────────────────────────────
# Bestehende Variablen (Rückwärtskompatibilität)
_viz        = CFG.get('visualisierung', {}).get('farben', {})
BG_DARK     = _viz.get('bg_dark',    '#0d1117')
BG_PANEL    = _viz.get('bg_panel',   '#141414')
C_PRICE     = _viz.get('c_price',    '#FFA726')
C_LOAD      = _viz.get('c_load',     '#66BB6A')
C_CHARGE    = _viz.get('c_charge',   '#1565C0')
C_FEED      = _viz.get('c_feed',     '#B71C1C')
SEG_COLORS  = _viz.get('seg_colors', ['#42A5F5', '#66BB6A', '#FFA726', '#EF5350'])
C_PRIV, C_GEW, C_IND, C_UTIL = SEG_COLORS

# UI-Strukturfarben
C_ACHSE      = _viz.get('c_achse',      '#aaaaaa')  # Achsenbeschriftungen
C_TICK       = _viz.get('c_tick',       '#bbbbbb')  # Tick-Labels
C_SPINE      = _viz.get('c_spine',      '#333333')  # Achsenrahmen
C_LEGENDE_BG = _viz.get('c_legende_bg', '#111111')  # Legenden-Hintergrund
C_GITTER     = _viz.get('c_gitter',     '#cccccc')  # Gitterlinien

# Funktionale Extrafarben (nur laden was das NB braucht)
C_DISPATCH   = _viz.get('c_dispatch',   '#AB47BC')  # Dispatch-optimal
C_STACKING   = _viz.get('c_stacking',   '#5DCAA5')  # Revenue Stacking
C_SOLAR      = _viz.get('c_solar',      '#FDD835')  # Solar-Ertrag
C_GRENZWERT  = _viz.get('c_amber_dark', '#FF6F00')  # Grenzwert / Warnung
C_CYAN       = _viz.get('c_cyan',       '#26C6DA')  # Flusswasser / Alt. Speicher
C_GRUEN_DARK = _viz.get('c_gruen_dark', '#388E3C')  # Erneuerbare

# Stilkonstanten
_stil               = CFG.get('visualisierung', {}).get('stil', {})
LW                  = _stil.get('linienbreite_standard', 1.5)   # Standard-Linienbreite
LW_DUENN            = _stil.get('linienbreite_duenn',    0.8)   # dünne Linien
LW_DICK             = _stil.get('linienbreite_dick',     2.5)   # dicke Linien
ALPHA_FLAECHE       = _stil.get('alpha_flaeche',         0.12)  # dezente Füllung
ALPHA_FLAECHE_STARK = _stil.get('alpha_flaeche_stark',   0.35)  # Balken / Füllung
ALPHA_LEGENDE       = _stil.get('alpha_legende',         0.30)  # Legenden-BG
ALPHA_GEDAEMPFT     = _stil.get('alpha_linie_gedaempft', 0.55)  # Nebenlinien
FS_TITEL            = _stil.get('schriftgroesse_titel',   13)   # Chart-Titel
FS_ACHSE            = _stil.get('schriftgroesse_achse',   10)   # Achsenbeschr.
FS_TICK             = _stil.get('schriftgroesse_tick',     9)   # Ticks
FS_LEGENDE          = _stil.get('schriftgroesse_legende',  8)   # Legende
FS_KLEIN            = _stil.get('schriftgroesse_klein',    7)   # Annotationen

# matplotlib rcParams — nur stabile, versionsunabhängige Keys (matplotlib >= 3.5)
# axes.titlecolor (3.8+) und axes.grid (stört Karten) bewusst NICHT gesetzt

mpl.rcParams.update({
    'figure.facecolor':  BG_DARK,
    'axes.facecolor':    BG_PANEL,
    'axes.edgecolor':    C_SPINE,
    'axes.labelcolor':   C_ACHSE,
    'axes.labelsize':    FS_ACHSE,
    'axes.titlesize':    FS_TITEL,
    'xtick.color':       C_TICK,
    'ytick.color':       C_TICK,
    'xtick.labelsize':   FS_TICK,
    'ytick.labelsize':   FS_TICK,
    'text.color':        'white',
    'lines.linewidth':   LW,
    'legend.facecolor':  C_LEGENDE_BG,
    'legend.framealpha': ALPHA_LEGENDE,
    'legend.fontsize':   FS_LEGENDE,
    'legend.edgecolor':  C_SPINE,
})
print('Farben & Stil geladen.')

print(f"Setup OK | Szenario={SZ_AKTIV} | Animationen → {CHARTS_DIR}")
# ── Animations-Parameter aus config.json (SSOT) ─────────────────────────────
# ── Animations-Parameter aus config.json (einzige Stelle) ───────────────────
_anim           = CFG.get('animation', {})
ANIM_DPI        = _anim.get('dpi', 110)
FRAMES_PER_WEEK = _anim.get('frames_per_week', 4)   # Frames je KW (Interpolation)
N_FRAMES_WEEK   = 52 * FRAMES_PER_WEEK              # Total Frames Jahresanimation
ANIM_FPS        = _anim.get('fps_chart', 10)         # fps → Loop = N_FRAMES_WEEK/fps
ANIM_EINZEL     = _anim.get('einzelbilder', True)
print(f'Animation: dpi={ANIM_DPI} | {FRAMES_PER_WEEK}f/Woche × 52 = {N_FRAMES_WEEK}f ')
print(f'           @{ANIM_FPS}fps = {N_FRAMES_WEEK/ANIM_FPS:.1f}s Loop | einzelbilder={ANIM_EINZEL}')


Farben & Stil geladen.
Setup OK | Szenario=realistisch | Animationen → ../output\charts\realistisch
Animation: dpi=110 | 4f/Woche × 52 = 208f 
           @10fps = 20.8s Loop | einzelbilder=True


**🔎 Quellcode der importierten lib-Funktionen**

Die Funktionen `should_skip` und `make_gif_chart` werden aus
`lib/plotting.py` importiert und in den folgenden Zellen verwendet.
Steuerung von `should_skip` über `sync/config.json` → `animation.modus`
und `animation.overrides`. `make_gif_chart` kapselt den PIL-GIF-Builder
inkl. Skip-Check, Einzelframe-Export und Fortschrittsausgabe.

Aufklappbar darunter sind die Quellcodes einsehbar.


In [17]:
show_source(should_skip)
show_source(make_gif_chart)


<details>
<summary>🔎 Quellcode: <code>should_skip</code> (aus <code>lib/plotting.py</code>)</summary>

```python
def should_skip(
    out_path: str,
    asset_type: str,
    name: str,
    cfg: dict[str, Any],
) -> bool:
    """Prüft anhand der Config, ob ein Chart/GIF neu erzeugt werden muss.

    Wird in jeder Zelle genutzt, die ein rechenintensives Artefakt erzeugt.
    Schema der relevanten Config-Sektion (in ``sync/config.json``):

    .. code-block:: json

        "animation": {
            "modus":          "skip_if_exists",
            "modus_statisch": "always",
            "overrides": {
                "kuer_k01_karte_verbraucher": "skip_if_exists"
            }
        }

    Entscheidungsreihenfolge
    ------------------------
    1. ``overrides[name]`` — explizit gesetzter Wert (höchste Priorität)
    2. ``asset_type == 'animation'`` → ``cfg['animation']['modus']``
       (Default: ``'skip_if_exists'``)
    3. ``asset_type == 'statisch'`` → ``cfg['animation']['modus_statisch']``
       (Default: ``'always'``). Wert ``'skip_if_exists_all'`` schaltet
       alle statischen Charts auf skip_if_exists um (Master-Schalter).

    Mode-Werte
    ----------
    * ``'skip_if_exists'``  — skip wenn Datei existiert und nicht leer
    * ``'always'``          — nie skip (Datei wird immer überschrieben)
    * ``'force_rebuild'``   — Alias für ``'always'``

    Parameter
    ---------
    out_path : str
        Zielpfad des zu erzeugenden Artefakts.
    asset_type : str
        Entweder ``'animation'`` oder ``'statisch'``.
    name : str
        Basename des Artefakts (ohne Extension), für ``overrides``-Lookup.
        Beispiel: ``'kuer_k04_anim_A_18h'``.
    cfg : dict
        Geladenes CFG-Dict aus ``sync/config.json``.

    Rückgabe
    --------
    bool
        ``True`` wenn die Erzeugung übersprungen werden kann, sonst ``False``.

    Beispiel
    --------
    >>> out = os.path.join(CHARTS_DIR, 'kuer_k04_anim_A_18h.gif')
    >>> if should_skip(out, 'animation', 'kuer_k04_anim_A_18h', CFG):
    ...     print(f"✓ Überspringe (existiert)")
    ... else:
    ...     make_gif_chart(...)
    """
    if asset_type not in ("animation", "statisch"):
        raise ValueError(
            f"asset_type muss 'animation' oder 'statisch' sein, "
            f"nicht {asset_type!r}"
        )

    anim_cfg = cfg.get("animation", {})
    overrides = anim_cfg.get("overrides", {})

    # 1. Explizit gesetzter Override (Keys mit _ sind Hinweis-Felder)
    if name in overrides and not name.startswith("_"):
        mode = overrides[name]
    # 2. Global je Asset-Typ
    elif asset_type == "animation":
        mode = anim_cfg.get("modus", "skip_if_exists")
    else:  # 'statisch'
        global_static = anim_cfg.get("modus_statisch", "always")
        # Master-Schalter: schaltet ALLE statischen Charts auf skip_if_exists
        if global_static == "skip_if_exists_all":
            mode = "skip_if_exists"
        else:
            mode = global_static

    # Validierung des Mode-Werts (nur informativ, fehlerhafte Werte → 'always')
    valid_modes = {"skip_if_exists", "always", "force_rebuild"}
    if mode not in valid_modes:
        # Nicht hart abbrechen — SSOT-Violation soll loud, aber nicht blockierend
        # sein, damit der Notebook-Lauf nicht an einem Tippfehler hängt.
        import warnings
        warnings.warn(
            f"Unbekannter Modus {mode!r} für {asset_type} '{name}'. "
            f"Erlaubt: {sorted(valid_modes)}. Fallback auf 'always'.",
            RuntimeWarning,
        )
        mode = "always"

    if mode == "skip_if_exists":
        return os.path.exists(out_path) and os.path.getsize(out_path) > 0
    # 'always' oder 'force_rebuild' → immer neu erzeugen
    return False
```

</details>


<details>
<summary>🔎 Quellcode: <code>make_gif_chart</code> (aus <code>lib/plotting.py</code>)</summary>

```python
def make_gif_chart(fig, update_fn, frames, fps, path,
                   dpi=None, save_frames=None, cfg=None):
    """PIL-basierter GIF-Builder für animierte Charts.

    Für jeden Frame-Wert wird ``update_fn(frame_val)`` aufgerufen, das Figure
    in PNG gerendert und in die Frame-Liste aufgenommen. Am Ende werden die
    Frames als animiertes GIF gespeichert (``loop=0``).

    Parameter
    ---------
    fig : matplotlib.figure.Figure
        Die Figure, die für jeden Frame neu gerendert wird. ``update_fn`` muss
        in-place die Axes-Elemente dieser Figure aktualisieren.
    update_fn : callable
        Funktion ``update_fn(frame_val)``, aktualisiert die dynamischen
        Chart-Elemente für den aktuellen Frame-Wert.
    frames : Iterable
        Sequenz der Frame-Werte (z.B. ``HOUR_TIMES`` für Tagesanimation oder
        ``WEEK_TIMES`` für Jahresanimation).
    fps : int
        Frames pro Sekunde im Output-GIF.
    path : str
        Zieldatei (``.gif``). Bei ``save_frames=True`` werden die Einzelframes
        zusätzlich in ``<path_ohne_ext>_frames/frame_NNNN.png`` gespeichert.
    dpi : int, optional
        DPI beim Rendering. Default: ``cfg['animation']['dpi']`` (wenn cfg
        gegeben), sonst 110.
    save_frames : bool, optional
        Einzelframes zusätzlich als PNG speichern. Default:
        ``cfg['animation']['einzelbilder']`` (wenn cfg gegeben), sonst False.
    cfg : dict, optional
        Geladenes ``sync/config.json``. Wenn gegeben:
          * Skip-Check via :func:`should_skip` (liest ``animation.modus`` und
            ``animation.overrides``)
          * dpi/save_frames-Defaults aus ``animation.dpi`` / ``animation.einzelbilder``

    Return
    ------
    None. Bei ``skip_if_exists`` + existierender Datei: return ohne Rendering.
    """
    import io as _io
    from PIL import Image as _PILImage

    # Skip-Check: wenn cfg gegeben und should_skip True → nichts tun
    if cfg is not None:
        _name = os.path.basename(path).rsplit('.', 1)[0]
        if should_skip(path, 'animation', _name, cfg):
            print(f'⏭️  {_name} übersprungen (existiert)')
            return

    # Defaults aus cfg oder Fallbacks
    if cfg is not None:
        _dpi    = dpi        if dpi        is not None else cfg.get('animation', {}).get('dpi', 110)
        _einzel = save_frames if save_frames is not None else cfg.get('animation', {}).get('einzelbilder', False)
    else:
        _dpi    = dpi        if dpi        is not None else 110
        _einzel = save_frames if save_frames is not None else False

    frame_dir = path.replace('.gif', '_frames')
    if _einzel:
        os.makedirs(frame_dir, exist_ok=True)

    # Frame-Loop
    frames_pil = []
    for i, frame_val in enumerate(frames):
        update_fn(frame_val)
        buf = _io.BytesIO()
        fig.savefig(buf, format='png', dpi=_dpi,
                    bbox_inches='tight', facecolor=fig.get_facecolor())
        buf.seek(0)
        img = _PILImage.open(buf).convert('RGB').copy()
        if _einzel:
            img.save(os.path.join(frame_dir, f'frame_{i:04d}.png'), optimize=True)
        frames_pil.append(img)

    if not frames_pil:
        print('⚠️  Keine Frames')
        return

    frames_pil[0].save(path, save_all=True, append_images=frames_pil[1:],
                       duration=int(1000 / fps), loop=0, optimize=True)
    n  = len(frames_pil)
    kb = os.path.getsize(path) // 1024
    print(f'✅ {os.path.basename(path)} ({n}f @{fps}fps={n/fps:.1f}s | {kb} KB)')
    if _einzel:
        print(f'   Frames: {frame_dir}/')
```

</details>


In [18]:
# ── GIF-Builder + Cubic-Interpolations-Engine für K_04 ──────────────────────
# Parameter aus config.json: ANIM_DPI, N_FRAMES_WEEK, ANIM_FPS, ANIM_EINZEL
import numpy as _np
from scipy.interpolate import CubicSpline as _CS

# Zeitachse: N_FRAMES_WEEK Punkte gleichmässig über 52 Wochen (1.0 → 52.0)
WEEK_TIMES = _np.linspace(1.0, 52.0, N_FRAMES_WEEK, endpoint=True)

def make_spline_w4(weeks_arr, values_arr, extrapolate=False):
    '''Cubic-Spline über wöchentliche Datenpunkte. Gibt Funktion f(t_w) zurück.'''
    cs = _CS(weeks_arr, values_arr, extrapolate=extrapolate)
    return cs

print(f"✅ GIF-Builder geladen | WEEK_TIMES: {WEEK_TIMES[0]:.1f}→{WEEK_TIMES[-1]:.1f} ({len(WEEK_TIMES)} Punkte)")


✅ GIF-Builder geladen | WEEK_TIMES: 1.0→52.0 (208 Punkte)


**⚙ Markdown-Prüfwerte (config-abhängig) und 📊 Markdown-Prüfwerte (transfer-abhängig)**:  
Werte mit ⚙ kommen aus `../sync/config.json`, Werte mit 📊 aus `../sync/transfer.json` (NB03-Output).  
Bei jeder Änderung dieser Quellen → Output mit ⚙/📊-Stellen im Markdown abgleichen.


In [19]:
# ── ⚙ Markdown-Prüfwerte (config-abhängig) ─────────────────────────────────
# Diese Werte erscheinen als ⚙ im Markdown-Text.
# Nach jeder config-Änderung: Ausgabe mit ⚙-Stellen im Text vergleichen!
print('=== ⚙ MARKDOWN-PRÜFWERTE (config-abhängig) ===')
print(f'  Animation: dpi={ANIM_DPI}, {FRAMES_PER_WEEK}f/Woche × 52 = {N_FRAMES_WEEK}f total @ {ANIM_FPS}fps')
print(f'  Loop-Dauer: {N_FRAMES_WEEK/ANIM_FPS:.1f}s | Einzelbilder-Export: {ANIM_EINZEL}')
print(f'  Skip-Modus (animation.modus): {CFG.get("animation", {}).get("modus", "?")}')


=== ⚙ MARKDOWN-PRÜFWERTE (config-abhängig) ===
  Animation: dpi=110, 4f/Woche × 52 = 208f total @ 10fps
  Loop-Dauer: 20.8s | Einzelbilder-Export: True
  Skip-Modus (animation.modus): skip_if_exists


In [20]:
# ── 📊 Markdown-Prüfwerte (transfer-abhängig) ──────────────────────────────
# Diese Werte erscheinen als 📊 im Markdown-Text.
# Nach jedem NB03-Lauf: Ausgabe mit 📊-Stellen im Text vergleichen!
TF      = load_transfer()
_tf_dz  = TF.get('datenzeitraum', {})
_tf_sai = TF.get('simulation', {}).get('saisonal', {})
print('=== 📊 MARKDOWN-PRÜFWERTE (transfer-abhängig) ===')
print(f'  Datenzeitraum         = {_tf_dz.get("start_date", "?")} bis {_tf_dz.get("end_date", "?")}')
if _tf_sai:
    print(f'  Saisonale Tages-Spread-Spannen (max−min, EUR/MWh):')
    for _s in ['Frühling', 'Sommer', 'Herbst', 'Winter']:
        if _s in _tf_sai:
            print(f'    {_s:<10}: {_tf_sai[_s]["spread_max_min_eur_mwh"]:>5.1f}')


=== 📊 MARKDOWN-PRÜFWERTE (transfer-abhängig) ===
  Datenzeitraum         = 2023 bis 2026
  Saisonale Tages-Spread-Spannen (max−min, EUR/MWh):
    Frühling  :  84.4
    Sommer    :  75.1
    Herbst    :  69.9
    Winter    :  49.0


---
### Daten laden

Spot-Preise und Netzlast aus `data/processed/` laden; wöchentliche Aggregation
der Stundenmittelwerte vorbereiten (Basis für alle drei Animationsoptionen).


In [8]:
# ── Preise und Last für Aggregation laden ────────────────────────────────────
CLEAN_FILE = os.path.join(DIR_PROC, 'ch_spot_prices_clean.csv')
LOAD_FILE  = os.path.join(DIR_RAW,  'ch_netzlast_raw.csv')

if not os.path.exists(CLEAN_FILE):
    print(f'⚠️  {CLEAN_FILE} fehlt → NB01/NB02 zuerst ausführen.')
    df_prices = None
elif not os.path.exists(LOAD_FILE):
    print(f'⚠️  {LOAD_FILE} fehlt → NB01 zuerst ausführen.')
    df_prices = None
else:
    df_prices = pd.read_csv(CLEAN_FILE, parse_dates=['timestamp'])
    df_prices['timestamp'] = pd.to_datetime(df_prices['timestamp'], utc=True)
    df_prices['hour'] = df_prices['timestamp'].dt.hour

    df_load = pd.read_csv(LOAD_FILE, parse_dates=['timestamp'])
    df_load['timestamp'] = pd.to_datetime(df_load['timestamp'], utc=True)
    df_load['hour'] = df_load['timestamp'].dt.hour

    print(f'df_prices : {df_prices.shape}')
    print(f'df_load   : {df_load.shape}')



df_prices : (29087, 6)


df_load   : (29070, 3)


---
## 1. Aggregation und Animations-Setup <a id='aggregation-und-animations-setup_K_04'></a>

[↑ Inhaltsverzeichnis](#toc_K_04)

52-Wochen-Index aufbauen, `matplotlib.animation` initialisieren;
gemeinsame Y-Achsengrenzen für frame-konsistente Darstellung berechnen.


In [9]:
if df_prices is None:
    print('⚠️  Daten fehlen — Setup übersprungen.')
    wh_price = wh_load = None
    CROSS_AVAIL = False
    WEEKS = []
    TAGESZEITEN = [0, 7, 12, 19]
else:
    # ── Saisonale Animationen: Setup ─────────────────────────────────────────────
    # 52 Kalenderwochen-Aggregate aus 2 Jahren (KW 1-52, gemittelt über beide Jahre)
    df_prices['week'] = df_prices['timestamp'].dt.isocalendar().week.astype(int)
    df_load['week']   = df_load['timestamp'].dt.isocalendar().week.astype(int)

    # Pro KW + Stunde: Ø Preis, Ø Last, Ø Import/Export
    # Hinweis: NaN-Werte entstehen bei echten Datenlücken in den ENTSO-E-Rohdaten
    # (z.B. API-Ausfälle, Fehlstunden). dropna() vor Aggregation verhindert dass
    # Wochen mit vollständig fehlenden Stunden NaN in mean/p25/p75 produzieren.
    # Wochen mit NaN-Aggregat werden in den Animationen still übersprungen.
    wh_price = (df_prices.dropna(subset=['price_eur_mwh'])
                .groupby(['week','hour'])['price_eur_mwh']
                .agg(mean='mean',
                     p25=lambda x: x.quantile(0.25),
                     p75=lambda x: x.quantile(0.75))
                .reset_index())
    wh_load  = (df_load.dropna(subset=['load_gw'])
                .groupby(['week', df_load['timestamp'].dt.hour])['load_gw']
                .mean().reset_index())
    wh_load.columns = ['week','hour','load_gw']

    # Import/Export falls vorhanden
    CROSS_FILE = os.path.join(DIR_INTER, 'ch_import_export_analyse.csv')  # berechnet in K_02
    if os.path.exists(CROSS_FILE):
        df_cx = pd.read_csv(CROSS_FILE, parse_dates=['timestamp'])
        df_cx['timestamp'] = pd.to_datetime(df_cx['timestamp'], utc=True)
        df_cx['week'] = df_cx['timestamp'].dt.isocalendar().week.astype(int)
        df_cx['hour'] = df_cx['timestamp'].dt.hour
        wh_cross = df_cx.groupby(['week','hour'])['net_export_mw'].mean().reset_index()
        CROSS_AVAIL = True
        print('Import/Export verfügbar')
    else:
        CROSS_AVAIL = False
        print('Import/Export nicht verfügbar – Animationen ohne Grenzflüsse')

    WEEKS       = list(range(1, 53))
    TAGESZEITEN = [0, 7, 12, 19]   # Stunden für Option A
    tick_kws    = [1, 5, 9, 13, 18, 22, 27, 31, 35, 40, 44, 48, 52]  # KW-Monatsanfänge

    # Hilfsfunktion: Saison-Farbe aus KW
    def kw_to_season_color(kw):
        if kw <= 8 or kw >= 49:  return C_PRIV      # Winter
        elif kw <= 21:            return C_LOAD       # Frühling
        elif kw <= 34:            return C_SOLAR      # Sommer
        else:                     return C_GRENZWERT  # Herbst

    def kw_to_month_label(kw):
        month = min(int((kw - 1) * 12 / 52) + 1, 12)
        return ['Jan','Feb','Mär','Apr','Mai','Jun',
                'Jul','Aug','Sep','Okt','Nov','Dez'][month-1]

    print(f'Animationsordner: {CHARTS_DIR}')
    print(f'Wochen: {len(WEEKS)} | Tageszeiten A: {TAGESZEITEN}')


Import/Export verfügbar
Animationsordner: ../output\charts\realistisch
Wochen: 52 | Tageszeiten A: [0, 7, 12, 19]


---
## 2. Verifikation <a id='verifikation_K_04'></a>

[↑ Inhaltsverzeichnis](#toc_K_04)

Aggregierten Datensatz auf Vollständigkeit prüfen: 52 Wochen × 24 Stunden;
Wertebereich auf Plausibilität kontrollieren.


In [10]:
if wh_price is None:
    print('⚠️  Übersprungen.')
else:
    # ── Verifikation: Aggregation auf NaN prüfen ────────────────────────────────
    nan_price = wh_price[['mean','p25','p75']].isnull().any(axis=1).sum()
    nan_load  = wh_load['load_gw'].isnull().sum()
    total_kw_h = len(wh_price)

    print(f'wh_price: {total_kw_h} KW×Stunden-Kombinationen')
    print(f'  NaN-Zeilen (mean/p25/p75): {nan_price}')
    if nan_price > 0:
        print('  Betroffene KW×Stunden:')
        print(wh_price[wh_price[['mean','p25','p75']].isnull().any(axis=1)]
              [['week','hour','mean','p25','p75']])
        print('  → Diese Frames werden in den Animationen übersprungen (NaN-Guard).')
    else:
        print('  Keine NaN — vollständige Abdeckung.')
    print(f'wh_load : NaN-Zeilen: {nan_load}')
    wh_price.head(3)


wh_price: 1248 KW×Stunden-Kombinationen
  NaN-Zeilen (mean/p25/p75): 0
  Keine NaN — vollständige Abdeckung.
wh_load : NaN-Zeilen: 0


---
## 3. Option A: Animation je Tageszeit (52 Wochen) <a id='option-a-animation-je-tageszeit-52-wochen_K_04'></a>

[↑ Inhaltsverzeichnis](#toc_K_04)

Erzeugt je eine GIF-Animation pro Tageszeit-Slot (z.B. 07h, 12h, 19h) —
zeigt wie sich der Preis zu einer fixen Stunde über das Jahr verändert.


In [11]:
if wh_price is None or not WEEKS:
    print('⚠️  Daten fehlen — übersprungen.')
else:
    # ── Option A: 4 GIFs je Tageszeit — cubic-interpoliert über 52 Wochen ────
    import math, numpy as _np_a
    n_tz = len(TAGESZEITEN)

    for t_idx, stunde in enumerate(TAGESZEITEN, 1):
        print(f'\n▶ Option A [{t_idx}/{n_tz}] {stunde:02d}:00 — {N_FRAMES_WEEK} Frames '
              f'({FRAMES_PER_WEEK}f/Woche @{ANIM_FPS}fps={N_FRAMES_WEEK/ANIM_FPS:.1f}s)...')

        _ph = wh_price[wh_price['hour']==stunde].set_index('week').sort_index()
        _lh = wh_load[wh_load['hour']==stunde].set_index('week').sort_index()
        _xh = (wh_cross[wh_cross['hour']==stunde].set_index('week').sort_index()
               if CROSS_AVAIL else None)

        # Splines über echte Wochendaten
        _wks = _np_a.array([w for w in WEEKS if w in _ph.index])
        _cs_mean = make_spline_w4(_wks, _ph.loc[_wks,'mean'].values)
        _cs_p25  = make_spline_w4(_wks, _ph.loc[_wks,'p25'].values)
        _cs_p75  = make_spline_w4(_wks, _ph.loc[_wks,'p75'].values)
        _cs_load = make_spline_w4(_wks, _lh.loc[[w for w in _wks if w in _lh.index],'load_gw'].values,) \
                   if len([w for w in _wks if w in _lh.index]) > 1 else None
        _cs_xval = (make_spline_w4(_wks, _xh.loc[[w for w in _wks if w in _xh.index],'net_export_mw'].values)
                    if _xh is not None and len([w for w in _wks if w in _xh.index])>1 else None)

        # Achslimits aus Rohdaten (nicht Spline-Extrapolation)
        p_min = _ph['p25'].min()*0.9; p_max = _ph['p75'].max()*1.1
        l_min = _lh['load_gw'].min()*0.9; l_max = _lh['load_gw'].max()*1.1
        x_min_v = (_xh['net_export_mw'].min()*1.1 if _xh is not None and _xh['net_export_mw'].min()<0 else -200)
        x_max_v = _xh['net_export_mw'].max()*1.1 if _xh is not None else 0

        fig_a, axes_a = plt.subplots(1, 3 if CROSS_AVAIL else 2,
                                     figsize=(13 if CROSS_AVAIL else 10, 5))
        fig_a.patch.set_facecolor(BG_DARK)
        for ax in axes_a: ax.set_facecolor(BG_PANEL); ax.tick_params(colors=C_TICK)
        for ax in axes_a:
            for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)
        ax_p, ax_l = axes_a[0], axes_a[1]
        ax_x = axes_a[2] if CROSS_AVAIL else None

        import matplotlib.ticker as _mt_a
        tick_kws_a = [1,5,9,13,18,22,27,31,35,40,44,48,52]
        for ax in [ax_p, ax_l] + ([ax_x] if ax_x else []):
            ax.set_xlim(0.5, 52.5)
            for kw_range, col in [((1,8),'#0a1525'),((9,21),'#0a1a0c'),
                                   ((22,34),'#1a1800'),((35,48),'#1a0e00'),((49,52),'#0a1525')]:
                ax.axvspan(kw_range[0]-0.5, kw_range[1]+0.5, alpha=0.3, color=col, zorder=0)
            ax.set_xticks(tick_kws_a)
            ax.set_xticklabels([kw_to_month_label(k) for k in tick_kws_a], fontsize=FS_LEGENDE)
            ax.grid(True, axis='y', alpha=ALPHA_FLAECHE)

        ax_p.set_ylim(p_min, p_max); ax_p.set_ylabel('EUR/MWh', color=C_ACHSE)
        ax_p.set_title(f'Spot-Preis {stunde:02d}:00', color='white', fontweight='bold')
        ax_l.set_ylim(l_min, l_max); ax_l.set_ylabel('GW', color=C_ACHSE)
        ax_l.set_title(f'Netzlast {stunde:02d}:00', color='white', fontweight='bold')
        ax_l.yaxis.set_major_formatter(_mt_a.FuncFormatter(lambda v,_: f'{v:.1f} GW'))
        if ax_x:
            ax_x.set_ylim(x_min_v, x_max_v); ax_x.set_ylabel('MW', color=C_ACHSE)
            ax_x.set_title(f'Netto-Export {stunde:02d}:00', color='white', fontweight='bold')
            ax_x.axhline(0, color='white', lw=LW_DUENN, linestyle='--')

        title_obj = fig_a.suptitle('', color='white', fontsize=12, fontweight='bold')
        # Historische Linien (wachsen mit jedem Frame)
        hist_line_p, = ax_p.plot([], [], color='#888', lw=LW_DUENN, alpha=0.5)
        hist_line_l, = ax_l.plot([], [], color='#888', lw=LW_DUENN, alpha=0.5)
        _eb_cont = [None]

        def update_a(t_w):
            # Interpolierter Zeitpunkt (1.0–52.0)
            w_int   = int(t_w)       # Letzte vollständige Woche
            col_cur = kw_to_season_color(max(1, w_int))
            season  = ('Winter' if w_int<=8 or w_int>=49 else
                       'Frühling' if w_int<=21 else 'Sommer' if w_int<=34 else 'Herbst')

            # Spline-Werte am aktuellen Zeitpunkt
            mean_p = float(_cs_mean(t_w))
            p25_v  = float(_cs_p25(t_w))
            p75_v  = float(_cs_p75(t_w))
            load_v = float(_cs_load(t_w)) if _cs_load else 0

            # Historische Linie: nur bis zur letzten vollen Woche
            hist_wks = [w for w in _wks if w <= w_int]
            if len(hist_wks) > 1:
                hist_line_p.set_data(hist_wks, [float(_cs_mean(w)) for w in hist_wks])
                hist_line_l.set_data(hist_wks, [float(_cs_load(w)) for w in hist_wks] if _cs_load else [])

            # Errorbar am aktuellen Zeitpunkt
            if _eb_cont[0] is not None:
                try: _eb_cont[0].remove()
                except Exception: pass
            lo = max(0.0, mean_p-p25_v); hi = max(0.0, p75_v-mean_p)
            _eb_cont[0] = ax_p.errorbar([t_w],[mean_p], yerr=[[lo],[hi]],
                                         fmt='o', color=col_cur, capsize=5,
                                         markersize=8, elinewidth=2, zorder=6)
            ax_l.plot([t_w],[load_v], 'o', color=col_cur, markersize=8, zorder=6)

            if ax_x and _cs_xval:
                xval = float(_cs_xval(t_w))
                ax_x.bar(t_w, xval, 0.3, color=C_LOAD if xval>=0 else C_UTIL, alpha=0.8, zorder=4)
                ax_x.set_title(f'Netto-Export {stunde:02d}:00', color='white', fontweight='bold')

            title_obj.set_text(
                f'{stunde:02d}:00 Uhr — KW {w_int:02d} ({kw_to_month_label(w_int)}, {season})'
                f'  |  Preis: {mean_p:.0f} EUR/MWh  |  Last: {load_v:.2f} GW')

        fname = os.path.join(CHARTS_DIR, f'kuer_k04_anim_A_{stunde:02d}h.gif')
        make_gif_chart(fig_a, update_a, WEEK_TIMES, ANIM_FPS, fname, cfg=CFG)
        plt.close(fig_a)

    print('\nOption A abgeschlossen')



▶ Option A [1/4] 00:00 — 208 Frames (4f/Woche @10fps=20.8s)...


✅ kuer_k04_anim_A_00h.gif (208f @10fps=20.8s | 2343 KB)
   Frames: ../output\charts\realistisch\kuer_k04_anim_A_00h_frames/

▶ Option A [2/4] 07:00 — 208 Frames (4f/Woche @10fps=20.8s)...


✅ kuer_k04_anim_A_07h.gif (208f @10fps=20.8s | 2521 KB)
   Frames: ../output\charts\realistisch\kuer_k04_anim_A_07h_frames/

▶ Option A [3/4] 12:00 — 208 Frames (4f/Woche @10fps=20.8s)...


✅ kuer_k04_anim_A_12h.gif (208f @10fps=20.8s | 2513 KB)
   Frames: ../output\charts\realistisch\kuer_k04_anim_A_12h_frames/

▶ Option A [4/4] 19:00 — 208 Frames (4f/Woche @10fps=20.8s)...


✅ kuer_k04_anim_A_19h.gif (208f @10fps=20.8s | 2572 KB)
   Frames: ../output\charts\realistisch\kuer_k04_anim_A_19h_frames/

Option A abgeschlossen


**🔎 Quellcode der importierten lib-Funktion**

`show_animation` aus `lib/widgets.py` — zeigt Animationen als GIF
(HTML-img) oder als interaktiven Slider. Unten aufklappbar der Quellcode.


In [11]:
show_source(show_animation)


<details>
<summary>🔎 Quellcode: <code>show_animation</code> (aus <code>lib/widgets.py</code>)</summary>

```python
def show_animation(
    path: str,
    mode: str = "gif",
    framerate: int = 10,
    width=1100,
    caption: str | None = None,
) -> None:
    """Zeigt eine Animation an — als GIF (inline) oder als Slider (interaktiv).

    Parameter
    ---------
    path : str
        Voller Pfad zur .gif-Datei. Bei mode='slider' wird der zugehörige
        ``<basename>_frames/``-Ordner automatisch gefunden.
    mode : {'gif', 'slider'}, default 'gif'
        ``'gif'``    — zeigt das fertige GIF inline
        ``'slider'`` — interaktiver Play/Slide-Viewer (Voraussetzung: ipywidgets
        installiert und Frames im _frames/-Ordner vorhanden)
    framerate : int, default 10
        fps bei mode='slider'. Bei mode='gif' ignoriert (fps steckt im GIF).
    width : int or str, default 1100
        Anzeigebreite. Int → Pixel (z.B. 900), Str → CSS (z.B. '100%', '900px').
    caption : str, optional
        Bildunterschrift. Wird nach der Animation ausgegeben.

    Beispiele
    ---------
    >>> show_animation('/output/charts/kuer_k04_anim_A.gif', mode='gif')
    >>> show_animation('/output/charts/kuer_k04_anim_A.gif', mode='slider',
    ...                framerate=15, width='100%')
    """
    from IPython.display import display, Image, HTML

    if mode not in ("gif", "slider"):
        raise ValueError(
            f"mode muss 'gif' oder 'slider' sein, nicht {mode!r}"
        )

    # Für beide Modi: Existenz prüfen (bei slider auf _frames-Ordner via slide_or_play)
    if mode == "gif":
        if not os.path.exists(path):
            print(f"⚠️  Datei nicht gefunden: {path}")
            return
        # Int width → px (Image-Parameter), Str width → CSS über HTML-img
        if isinstance(width, int):
            display(Image(filename=path, width=width))
        else:
            display(HTML(f'<img src="{path}" style="width:{width};display:block">'))
    else:  # mode == 'slider'
        # Breite als CSS-String für slide_or_play
        if isinstance(width, int):
            width_css = f"{width}px"
        else:
            width_css = width
        slide_or_play(path, framerate=framerate, image_width=width_css)

    if caption:
        print(f"\n{caption}\n")
```

</details>


In [26]:
# ── Animation anzeigen ──────────────────────────────────────────────────
fname = os.path.join(CHARTS_DIR, 'kuer_k04_anim_A_12h.gif')
show_animation(fname, mode='slider', framerate=10, width=700)


📼 Lade 208 Frames aus ../output\charts\realistisch\kuer_k04_anim_A_12h_frames...
   ✅ geladen (16 MB RAM)


---
## 4. Option B: 4-Panel-Animation <a id='option-b-4-panel-animation_K_04'></a>

[↑ Inhaltsverzeichnis](#toc_K_04)

Alle vier Tageszeit-Slots in einem 2×2-Grid animiert — direkter Vergleich
von Morgen-, Mittags-, Abend- und Nachtprofil über 52 Wochen.


In [14]:
if wh_price is None or not WEEKS:
    print('⚠️  Daten fehlen — übersprungen.')
else:
    # ── Option B: 4-Panel — cubic-interpoliert ────────────────────────────────
    import numpy as _np_b, matplotlib.ticker as _mt_b
    print(f'\n▶ Option B: 4-Panel — {N_FRAMES_WEEK} Frames @{ANIM_FPS}fps={N_FRAMES_WEEK/ANIM_FPS:.1f}s...')

    STUNDEN_B = [0, 7, 12, 19]
    STUNDEN_LABELS = ['00:00 (Nacht)','07:00 (Morgen)','12:00 (Mittag)','19:00 (Abend)']
    STUNDEN_COLORS = [C_PRIV, C_SOLAR, C_GRENZWERT, C_UTIL]

    _wks_b = _np_b.array(sorted(WEEKS))
    _splines_b = {}
    for s in STUNDEN_B:
        _ph = wh_price[wh_price['hour']==s].set_index('week').sort_index()
        _lh = wh_load[wh_load['hour']==s].set_index('week').sort_index()
        _wk_s = _np_b.array([w for w in _wks_b if w in _ph.index])
        _splines_b[s] = {
            'mean': make_spline_w4(_wk_s, _ph.loc[_wk_s,'mean'].values),
            'p25':  make_spline_w4(_wk_s, _ph.loc[_wk_s,'p25'].values),
            'p75':  make_spline_w4(_wk_s, _ph.loc[_wk_s,'p75'].values),
            'load': make_spline_w4(_wk_s, _lh.loc[[w for w in _wk_s if w in _lh.index],'load_gw'].values)
                   if len([w for w in _wk_s if w in _lh.index])>1 else None,
            'wks':  _wk_s,
        }
        _splines_b[s]['plim'] = (_ph['p25'].min()*0.9, _ph['p75'].max()*1.1)
        _splines_b[s]['llim'] = (_lh['load_gw'].min()*0.95, _lh['load_gw'].max()*1.05)

    fig_b, axes_b = plt.subplots(2, 4, figsize=(22, 9))
    fig_b.patch.set_facecolor(BG_DARK)
    for ax in axes_b.flat:
        ax.set_facecolor(BG_PANEL); ax.tick_params(colors=C_TICK)
        for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)

    tick_kws_b = [1,5,9,13,18,22,27,31,35,40,44,48,52]
    for col_idx,(s,lbl,col) in enumerate(zip(STUNDEN_B,STUNDEN_LABELS,STUNDEN_COLORS)):
        for row_idx in range(2):
            ax = axes_b[row_idx,col_idx]; ax.set_xlim(0.5,52.5)
            if row_idx==0:
                ax.set_ylim(*_splines_b[s]['plim'])
                ax.set_title(lbl, color=col, fontsize=FS_ACHSE, fontweight='bold')
                if col_idx==0: ax.set_ylabel('EUR/MWh', color=C_ACHSE)
            else:
                ax.set_ylim(*_splines_b[s]['llim'])
                ax.set_xlabel('Monat', color=C_ACHSE)
                ax.yaxis.set_major_formatter(_mt_b.FuncFormatter(lambda v,_: f'{v:.1f}'))
                if col_idx==0: ax.set_ylabel('GW', color=C_ACHSE)
            ax.set_xticks(tick_kws_b)
            ax.set_xticklabels([kw_to_month_label(k) for k in tick_kws_b], fontsize=FS_KLEIN)
            ax.grid(True, axis='y', alpha=0.10)
            for kw_range,bg in [((1,8),'#0a1525'),((9,21),'#0a1a0c'),
                                 ((22,34),'#1a1800'),((35,48),'#1a0e00'),((49,52),'#0a1525')]:
                ax.axvspan(kw_range[0]-.5, kw_range[1]+.5, alpha=0.25, color=bg, zorder=0)

    plt.tight_layout(rect=[0,0,1,0.95])
    title_b = fig_b.suptitle('', color='white', fontsize=FS_TITEL, fontweight='bold')

    def update_b(t_w):
        w_int   = int(t_w)
        col_cur = kw_to_season_color(max(1,w_int))
        season  = ('Winter' if w_int<=8 or w_int>=49 else
                   'Frühling' if w_int<=21 else 'Sommer' if w_int<=34 else 'Herbst')
        title_b.set_text(f'Saisonaler Verlauf — KW {w_int:02d} ({kw_to_month_label(w_int)}, {season})')
        for col_idx,(s,col_s) in enumerate(zip(STUNDEN_B,STUNDEN_COLORS)):
            sp = _splines_b[s]
            mean_p = float(sp['mean'](t_w)); p25_v=float(sp['p25'](t_w)); p75_v=float(sp['p75'](t_w))
            load_v = float(sp['load'](t_w)) if sp['load'] else 0
            ax_price=axes_b[0,col_idx]; ax_load=axes_b[1,col_idx]
            # Historische Linie bis zur letzten vollen Woche
            hist_wks = [w for w in sp['wks'] if w<=w_int]
            if len(hist_wks)>1:
                ax_price.plot(hist_wks,[float(sp['mean'](w)) for w in hist_wks],
                              color=col_s,lw=0.9,alpha=0.35,zorder=3)
                if sp['load']:
                    ax_load.plot(hist_wks,[float(sp['load'](w)) for w in hist_wks],
                                 color=col_s,lw=0.9,alpha=0.35,zorder=3)
            # Aktueller Punkt (interpoliert)
            ax_price.errorbar([t_w],[mean_p],
                               yerr=[[max(0,mean_p-p25_v)],[max(0,p75_v-mean_p)]],
                               fmt='o',color=col_cur,capsize=4,markersize=7,elinewidth=1.8,zorder=6)
            ax_load.plot([t_w],[load_v],'o',color=col_cur,markersize=7,zorder=6)

    fname_b = os.path.join(CHARTS_DIR, 'kuer_k04_anim_B_4panel.gif')
    make_gif_chart(fig_b, update_b, WEEK_TIMES, ANIM_FPS, fname_b, cfg=CFG)
    plt.close(fig_b)



▶ Option B: 4-Panel — 208 Frames @10fps=20.8s...


✅ kuer_k04_anim_B_4panel.gif (208f @10fps=20.8s | 4208 KB)
   Frames: ../output\charts\realistisch\kuer_k04_anim_B_4panel_frames/


In [29]:
# ── Animation anzeigen ──────────────────────────────────────────────────
fname_b = os.path.join(CHARTS_DIR, 'kuer_k04_anim_B_4panel.gif')
show_animation(fname_b, mode='slider', framerate=10, width=700)


📼 Lade 208 Frames aus ../output\charts\realistisch\kuer_k04_anim_B_4panel_frames...
   ✅ geladen (40 MB RAM)


---
## 5. Option C: Spread-Animation rollierend <a id='option-c-spread-animation-rollierend_K_04'></a>

[↑ Inhaltsverzeichnis](#toc_K_04)

Rollierender 4-Wochen-Arbitrage-Spread animiert über das Jahr —
zeigt saisonal optimale Einzeitraum-Fenster für den Batterie-[Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch).


In [16]:
if wh_price is None or not WEEKS:
    print('⚠️  Daten fehlen — übersprungen.')
else:
    # ── Option C: Spread — cubic-interpoliert ─────────────────────────────────
    import numpy as _np_c
    print(f'\n▶ Option C: Spread — {N_FRAMES_WEEK} Frames @{ANIM_FPS}fps={N_FRAMES_WEEK/ANIM_FPS:.1f}s...')

    if 'tick_kws' not in dir(): tick_kws = [1,5,9,13,18,22,27,31,35,40,44,48,52]

    week_q      = df_prices.groupby('week')['price_eur_mwh'].quantile([0.25,0.75]).unstack()
    week_q.columns = ['p25','p75']
    week_agg    = df_prices.groupby('week')['price_eur_mwh'].agg(mean='mean',
                     neg_pct=lambda x:(x<0).mean()*100)
    week_spread = week_q.join(week_agg).reset_index()
    week_spread['spread'] = week_spread['p75']-week_spread['p25']

    df_prices['day'] = df_prices['timestamp'].dt.date
    _day_q = df_prices.groupby('day')['price_eur_mwh'].quantile([0.25,0.75]).unstack()
    _day_q.columns = ['p25_day','p75_day']
    df_prices = df_prices.join(_day_q, on='day')
    df_prices['dispatch_load'] = df_prices['price_eur_mwh']<=df_prices['p25_day']
    df_prices['dispatch_feed'] = df_prices['price_eur_mwh']>=df_prices['p75_day']
    week_dispatch = (df_prices.groupby('week')[['dispatch_load','dispatch_feed']].mean()*100).reset_index()
    week_spread   = week_spread.merge(week_dispatch, on='week')
    ws_idx        = week_spread.set_index('week').sort_index()

    _wks_c = _np_c.array(sorted(ws_idx.index.tolist()))
    _cs_spread = make_spline_w4(_wks_c, ws_idx.loc[_wks_c,'spread'].values)
    _cs_neg    = make_spline_w4(_wks_c, ws_idx.loc[_wks_c,'neg_pct'].values)
    _cs_dload  = make_spline_w4(_wks_c, ws_idx.loc[_wks_c,'dispatch_load'].values*24/100)
    _cs_dfeed  = make_spline_w4(_wks_c, ws_idx.loc[_wks_c,'dispatch_feed'].values*24/100)

    fig_c, axes_c = plt.subplots(3,1,figsize=(14,12),sharex=True)
    fig_c.patch.set_facecolor(BG_DARK)
    for ax in axes_c:
        ax.set_facecolor(BG_PANEL); ax.tick_params(colors=C_TICK)
        for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)
        ax.set_xlim(0.5,52.5); ax.grid(True,axis='y',alpha=ALPHA_FLAECHE)
        for kw_range,col_bg in [((1,8),'#0a1525'),((9,21),'#0a1a0c'),
                                 ((22,34),'#1a1800'),((35,48),'#1a0e00'),((49,52),'#0a1525')]:
            ax.axvspan(kw_range[0]-.5,kw_range[1]+.5,alpha=0.25,color=col_bg,zorder=0)

    axes_c[0].set_ylim(0, week_spread['spread'].max()*1.15)
    axes_c[0].set_ylabel('EUR/MWh',color=C_ACHSE)
    axes_c[0].set_title('Wöchentlicher Arbitrage-Spread (p75−p25)',color='white',fontweight='bold')
    axes_c[0].axhline(week_spread['spread'].mean(),color='#888',lw=1,linestyle=':')
    axes_c[1].set_ylim(0, max(week_spread['neg_pct'].max()*1.2,0.1))
    axes_c[1].set_ylabel('%',color=C_ACHSE)
    axes_c[1].set_title('Negativpreis-Anteil [%]',color='white',fontweight='bold')
    axes_c[2].set_ylim(0,30); axes_c[2].set_ylabel('%',color=C_ACHSE)
    axes_c[2].set_xlabel('Monat',color=C_ACHSE)
    axes_c[2].set_title('Dispatch-Stunden [%]',color='white',fontweight='bold')
    axes_c[2].set_xticks(tick_kws)
    axes_c[2].set_xticklabels([kw_to_month_label(k) for k in tick_kws])

    # Kontinuierliche Spline-Kurve als Hintergrundlinie (alle 52 Wochen)
    _t_full = _np_c.linspace(1,52,500)
    axes_c[0].plot(_t_full, _cs_spread(_t_full), color=C_PRICE, lw=0.6, alpha=0.25, zorder=1)
    axes_c[1].plot(_t_full, _cs_neg(_t_full),    color=C_UTIL,  lw=0.6, alpha=0.25, zorder=1)

    title_c = fig_c.suptitle('', color='white', fontsize=FS_TITEL, fontweight='bold')
    # Laufende Linien bis aktuellem Frame
    line_spread, = axes_c[0].plot([],[], color=C_PRICE, lw=LW_DICK, zorder=4)
    line_neg,    = axes_c[1].plot([],[], color=C_UTIL,  lw=LW_DICK, zorder=4)

    def update_c(t_w):
        w_int   = int(t_w)
        col_cur = kw_to_season_color(max(1,w_int))
        season  = ('Winter' if w_int<=8 or w_int>=49 else
                   'Frühling' if w_int<=21 else 'Sommer' if w_int<=34 else 'Herbst')

        # Linie von Woche 1 bis aktuellen Zeitpunkt (interpoliert)
        _t_hist = _np_c.linspace(1.0, t_w, max(2, int((t_w-1)*FRAMES_PER_WEEK)+2))
        line_spread.set_data(_t_hist, _cs_spread(_t_hist))
        line_neg.set_data(_t_hist, _cs_neg(_t_hist))

        sp_val = float(_cs_spread(t_w)); ng_val = float(_cs_neg(t_w))
        ld_val = float(_cs_dload(t_w));  fd_val = float(_cs_dfeed(t_w))

        # Aktueller Punkt (nur sichtbar wenn Woche vollständig)
        if w_int >= 1:
            axes_c[0].plot([t_w],[sp_val],'o',color=col_cur,markersize=8,zorder=6)
            axes_c[1].plot([t_w],[ng_val],'o',color=col_cur,markersize=8,zorder=6)
            axes_c[2].bar(t_w, ld_val, 0.3, color=C_CHARGE, alpha=0.8, zorder=4)
            axes_c[2].bar(t_w, fd_val, 0.3, bottom=ld_val, color=C_FEED, alpha=0.8, zorder=4)

        title_c.set_text(f'Arbitrage-Kennzahlen KW {w_int:02d} ({kw_to_month_label(w_int)}, '
                         f'{season})  |  Spread: {sp_val:.0f} EUR/MWh  |  Negativ: {ng_val:.1f}%')

    fname_c = os.path.join(CHARTS_DIR, 'kuer_k04_anim_C_spread.gif')
    make_gif_chart(fig_c, update_c, WEEK_TIMES, ANIM_FPS, fname_c, cfg=CFG)
    plt.close(fig_c)



▶ Option C: Spread — 208 Frames @10fps=20.8s...


✅ kuer_k04_anim_C_spread.gif (208f @10fps=20.8s | 4244 KB)
   Frames: ../output\charts\realistisch\kuer_k04_anim_C_spread_frames/


In [31]:
# ── Animation anzeigen ──────────────────────────────────────────────────
fname_c = os.path.join(CHARTS_DIR, 'kuer_k04_anim_C_spread.gif')
show_animation(fname_c, mode='slider', framerate=10, width=700)


📼 Lade 208 Frames aus ../output\charts\realistisch\kuer_k04_anim_C_spread_frames...
   ✅ geladen (30 MB RAM)


---

---
## Fazit <a id='fazit_K_04'></a>

[↑ Inhaltsverzeichnis](#toc_K_04)

Drei Animations-Typen erzeugt. Visuelle Bestätigung der quantitativen Ergebnisse
aus NB03/NB04:

- **Frühling hat ~1.7× breitere Spread-Kanäle als Winter** (84 EUR/MWh📊 vs. 49 EUR/MWh📊 Tages-Spread-Spanne) — wirtschaftliches Arbitrage-Fenster im Jahresverlauf deutlich asymmetrisch verteilt
- **Sommer (75 EUR/MWh📊)** zeigt das Solar-Mittagstief klar als tägliche Delle im Preisband — trotz häufiger Negativpreise bleibt die Tagesspanne hoch (Mittagstief vs. Abendspitze)
- **Herbst (70 EUR/MWh📊)** als Übergangsphase — moderate Spannen ganzjährig

Die GIFs werden in K_00 (Business Strategy) als Anschauungsmaterial referenziert.


---
## Potenzielle Erweiterungen <a id='erweiterungen_K_04'></a>

### ipywidgets: Einzelframe-Navigation

Die erzeugten GIFs laufen automatisch durch — ein Pausieren oder gezieltes
Anspringen einzelner Frames ist nicht möglich. Mit `ipywidgets` liesse sich
ein interaktiver Slider ergänzen, der denselben Datensatz Frame für Frame
durchblättert:

```python
# Skizze — nicht produktiv
import ipywidgets as widgets
from IPython.display import display

@widgets.interact(woche=widgets.IntSlider(min=1, max=52, step=1, value=1,
                                          description='KW:'))
def show_week(woche):
    frame = wh_price[wh_price['week'] == woche]
    # ... Plot analog zu Animation C
```

> **Hinweis:** `ipywidgets` erfordert einen laufenden Kernel — funktioniert nicht in statischem HTML-Export oder `run_all`. Daher vorerst nicht implementiert.

### ipywidgets: Produktkonfigurator (K_10)

In `K_10_Produkt_Steckbrief` werden Parameter wie Kapazität, Leistung und
Verkaufspreis manuell in einer Konfigurationszelle gesetzt. Slider würden
ROI und Payback live neu berechnen — sinnvoll für Kundenpräsentationen,
aber inkompatibel mit `run_all` und `transfer.json`-Pipeline.
Gleiche Einschränkung wie oben.


---
## Abschluss <a id='abschluss_K_04'></a>

[↑ Inhaltsverzeichnis](#toc_K_04)

### Erzeugte GIF-Animationen

Die drei Animationsoptionen (A/B/C) sind komplementär:

| Option | Format | Inhalt | Empfohlen für |
|--------|--------|--------|---------------|
| **A** — Je Tageszeit | 4× GIF, 208 Frames⚙ (4f/Woche × 52) | Preis/Last-Verlauf für 00/07/12/19 Uhr über das Jahr | Vertiefungsanalyse |
| **B** — 4-Panel | 1× GIF | Alle vier Tageszeiten simultan | Präsentation |
| **C** — Spread rollierend | 1× GIF | Wöchentlicher Spread-Aufbau + Negativpreis-Anteil | Bericht K_00 |



**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `final_check` wird aus `lib/io_ops.py` importiert und
in der folgenden Zelle verwendet. Aufklappbar ist der Quellcode einsehbar


In [18]:
show_source(final_check)

<details>
<summary>🔎 Quellcode: <code>final_check</code> (aus <code>lib/io_ops.py</code>)</summary>

```python
def final_check(nb_label, files=None, *, weiter_msg=None, fehler_msg=None,
                extras=None, show_dataindex=False,
                dataindex_path='../sync/dataindex.csv', width=60):
    """Standardisierte End-of-Notebook-Kontrolle für Pflicht- und Kür-NBs.

    Prüft Existenz und Mindestgrösse der angegebenen Output-Dateien,
    gibt formatiertes Resultat aus und liefert ``all_ok`` als Bool zurück.

    Parameter
    ---------
    nb_label : str
        Label des Notebooks im Output-Header, z.B. ``"NB01"``, ``"K_03"``.
    files : list of tuple, optional
        Zu prüfende Dateien als ``(path, label, min_bytes)``-Tuples.

        * ``min_bytes = 0`` → nur Existenz prüfen, Grösse nicht ausgeben
          (z.B. für PNG-Charts).
        * ``min_bytes > 0`` → zusätzlich Grösse prüfen und in KB/MB ausgeben
          (z.B. für CSV-Dateien).

        Bei ``files=None`` oder ``files=[]`` wird kein Check ausgeführt;
        die Funktion dient dann als reiner Status-Print (für Report-NBs
        ohne eigene Outputs wie K_00).
    weiter_msg : str, optional
        Nachricht für den Erfolgsfall, z.B. ``"NB02 Daten Bereinigung"``.
        Default: ``"nächstes Notebook"``.
    fehler_msg : str, optional
        Nachricht für den Fehlerfall (Kurzform, ohne "Fehler beheben vor").
        Default: identisch mit ``weiter_msg``.
    extras : list of str, optional
        Zusätzliche Print-Zeilen zwischen Datei-Check und Weiter-/Fehler-Hinweis.
        Sinnvoll für Kür-Hinweise oder Kontext.
    show_dataindex : bool, default False
        Wenn True, wird der aktive Auszug aus ``../sync/dataindex.csv`` ausgegeben.
        Typisch für NB01.
    dataindex_path : str, default '../sync/dataindex.csv'
        Pfad zur dataindex.csv (für ``show_dataindex=True``).
    width : int, default 60
        Breite der Trennlinie aus ``=``-Zeichen.

    Return
    ------
    bool
        ``True`` wenn alle Files existieren und Mindestgrösse erfüllen,
        ``False`` sonst. Bei ``files=None``/leer immer ``True``.
    """
    print(f'{nb_label} – Abschlusskontrolle')
    print('=' * width)

    all_ok = True

    if files:
        for path, label, min_bytes in files:
            exists = os.path.exists(path)
            size = os.path.getsize(path) if exists else 0
            ok = exists and size >= min_bytes

            if min_bytes > 0:
                size_str = _format_size(size) if exists else '   FEHLT'
                print(f'  {"✅" if ok else "❌"}  {label:<45} {size_str}')
            else:
                print(f'  {"✅" if ok else "❌"}  {label}')

            if not ok:
                all_ok = False

    if extras:
        if files:
            print()
        for line in extras:
            print(line)

    if show_dataindex and os.path.exists(dataindex_path):
        import pandas as pd
        df_idx = pd.read_csv(dataindex_path)
        active = df_idx[df_idx['status'] == 'active']
        print(f'\ndataindex.csv: {len(df_idx)} Einträge total, {len(active)} active')
        print(active[['filename', 'data_type', 'rows', 'size_kb', 'timestamp']]
              .to_string(index=False))

    print()
    weiter = weiter_msg or 'nächstes Notebook'
    fehler = fehler_msg or weiter
    if all_ok:
        print(f'→ Weiter mit {weiter}.')
    else:
        print(f'→ Fehler beheben vor {fehler}.')

    return all_ok
```

</details>


In [19]:
show_source(final_check)

<details>
<summary>🔎 Quellcode: <code>final_check</code> (aus <code>lib/io_ops.py</code>)</summary>

```python
def final_check(nb_label, files=None, *, weiter_msg=None, fehler_msg=None,
                extras=None, show_dataindex=False,
                dataindex_path='../sync/dataindex.csv', width=60):
    """Standardisierte End-of-Notebook-Kontrolle für Pflicht- und Kür-NBs.

    Prüft Existenz und Mindestgrösse der angegebenen Output-Dateien,
    gibt formatiertes Resultat aus und liefert ``all_ok`` als Bool zurück.

    Parameter
    ---------
    nb_label : str
        Label des Notebooks im Output-Header, z.B. ``"NB01"``, ``"K_03"``.
    files : list of tuple, optional
        Zu prüfende Dateien als ``(path, label, min_bytes)``-Tuples.

        * ``min_bytes = 0`` → nur Existenz prüfen, Grösse nicht ausgeben
          (z.B. für PNG-Charts).
        * ``min_bytes > 0`` → zusätzlich Grösse prüfen und in KB/MB ausgeben
          (z.B. für CSV-Dateien).

        Bei ``files=None`` oder ``files=[]`` wird kein Check ausgeführt;
        die Funktion dient dann als reiner Status-Print (für Report-NBs
        ohne eigene Outputs wie K_00).
    weiter_msg : str, optional
        Nachricht für den Erfolgsfall, z.B. ``"NB02 Daten Bereinigung"``.
        Default: ``"nächstes Notebook"``.
    fehler_msg : str, optional
        Nachricht für den Fehlerfall (Kurzform, ohne "Fehler beheben vor").
        Default: identisch mit ``weiter_msg``.
    extras : list of str, optional
        Zusätzliche Print-Zeilen zwischen Datei-Check und Weiter-/Fehler-Hinweis.
        Sinnvoll für Kür-Hinweise oder Kontext.
    show_dataindex : bool, default False
        Wenn True, wird der aktive Auszug aus ``../sync/dataindex.csv`` ausgegeben.
        Typisch für NB01.
    dataindex_path : str, default '../sync/dataindex.csv'
        Pfad zur dataindex.csv (für ``show_dataindex=True``).
    width : int, default 60
        Breite der Trennlinie aus ``=``-Zeichen.

    Return
    ------
    bool
        ``True`` wenn alle Files existieren und Mindestgrösse erfüllen,
        ``False`` sonst. Bei ``files=None``/leer immer ``True``.
    """
    print(f'{nb_label} – Abschlusskontrolle')
    print('=' * width)

    all_ok = True

    if files:
        for path, label, min_bytes in files:
            exists = os.path.exists(path)
            size = os.path.getsize(path) if exists else 0
            ok = exists and size >= min_bytes

            if min_bytes > 0:
                size_str = _format_size(size) if exists else '   FEHLT'
                print(f'  {"✅" if ok else "❌"}  {label:<45} {size_str}')
            else:
                print(f'  {"✅" if ok else "❌"}  {label}')

            if not ok:
                all_ok = False

    if extras:
        if files:
            print()
        for line in extras:
            print(line)

    if show_dataindex and os.path.exists(dataindex_path):
        import pandas as pd
        df_idx = pd.read_csv(dataindex_path)
        active = df_idx[df_idx['status'] == 'active']
        print(f'\ndataindex.csv: {len(df_idx)} Einträge total, {len(active)} active')
        print(active[['filename', 'data_type', 'rows', 'size_kb', 'timestamp']]
              .to_string(index=False))

    print()
    weiter = weiter_msg or 'nächstes Notebook'
    fehler = fehler_msg or weiter
    if all_ok:
        print(f'→ Weiter mit {weiter}.')
    else:
        print(f'→ Fehler beheben vor {fehler}.')

    return all_ok
```

</details>


In [20]:
# ── Abschlusskontrolle K_04 ─────────────────────────────────────────────────
final_check(
    'K_04',
    files=[
        (os.path.join(CHARTS_DIR, 'kuer_k04_anim_A_00h.gif'),    'Animation A: 00:00-Slot über 52 Wochen', 100_000),
        (os.path.join(CHARTS_DIR, 'kuer_k04_anim_A_07h.gif'),    'Animation A: 07:00-Slot über 52 Wochen', 100_000),
        (os.path.join(CHARTS_DIR, 'kuer_k04_anim_A_12h.gif'),    'Animation A: 12:00-Slot über 52 Wochen', 100_000),
        (os.path.join(CHARTS_DIR, 'kuer_k04_anim_A_19h.gif'),    'Animation A: 19:00-Slot über 52 Wochen', 100_000),
        (os.path.join(CHARTS_DIR, 'kuer_k04_anim_B_4panel.gif'), 'Animation B: 4-Panel-Vergleich', 100_000),
        (os.path.join(CHARTS_DIR, 'kuer_k04_anim_C_spread.gif'), 'Animation C: Spread rollierend', 100_000),
    ],
    weiter_msg='K_05/K_99 oder K_00 (Business Strategy)',
)


K_04 – Abschlusskontrolle
  ✅  Animation A: 00:00-Slot über 52 Wochen            2.3 MB
  ✅  Animation A: 07:00-Slot über 52 Wochen            2.5 MB
  ✅  Animation A: 12:00-Slot über 52 Wochen            2.5 MB
  ✅  Animation A: 19:00-Slot über 52 Wochen            2.5 MB
  ✅  Animation B: 4-Panel-Vergleich                    4.1 MB
  ✅  Animation C: Spread rollierend                    4.1 MB

→ Weiter mit K_05/K_99 oder K_00 (Business Strategy).


True

| [← K_03 – Marktdynamik](K_03_Marktdynamik.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_05 – Revenue Stacking →](K_05_Revenue_Stacking.ipynb) |
|:---|:---:|---:|